# Scaled Dot-Product Attention - End-to-End Pipeline
**Date**: 2026-05-31  
**Objective**: Build a self-contained notebook that demonstrates scaled dot-product attention on a toy text classification task.

## 1) Imports and Setup

In [ ]:
from __future__ import annotations

import importlib.util
import math
import os
import random
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

SEED: int = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)


def parse_use_gpu_flag(raw_value: str) -> bool:
    normalized = raw_value.strip().lower()
    return normalized not in {"0", "false", "no", "off"}


def load_runtime_env() -> None:
    env_files = [
        Path("configs/runtime.env"),
        Path("configs/runtime.env.example"),
        Path("../configs/runtime.env"),
        Path("../configs/runtime.env.example"),
    ]
    for env_file in env_files:
        if env_file.exists():
            for line in env_file.read_text(encoding="utf-8").splitlines():
                line = line.strip()
                if not line or line.startswith("#") or "=" not in line:
                    continue
                key, value = line.split("=", 1)
                os.environ.setdefault(key.strip(), value.strip().strip("\"'"))
            break


load_runtime_env()
USE_GPU = parse_use_gpu_flag(os.getenv("USE_GPU", "1"))

_torch_cuda = False
_tf_gpu = False
if importlib.util.find_spec("torch") is not None:
    import torch

    _torch_cuda = torch.cuda.is_available()
if importlib.util.find_spec("tensorflow") is not None:
    import tensorflow as tf

    tf.random.set_seed(SEED)
    _tf_gpu = bool(tf.config.list_physical_devices("GPU"))

runtime_device = "cuda" if USE_GPU and (_torch_cuda or _tf_gpu) else "cpu"
print(f"USE_GPU={int(USE_GPU)} | runtime_device={runtime_device}")

## 2) Configuration and Constants

In [ ]:
@dataclass(frozen=True)
class ExperimentConfig:
    num_samples: int = 256
    seq_len: int = 24
    d_model: int = 32
    num_heads: int = 4
    num_classes: int = 2
    learning_rate: float = 0.2
    epochs: int = 120
    attention_name: str = "Scaled Dot-Product Attention"


CFG = ExperimentConfig()
CFG

## 3) Data Loading

In [ ]:
POSITIVE_TERMS = ["good", "great", "fast", "stable", "accurate"]
NEGATIVE_TERMS = ["bad", "slow", "broken", "noisy", "weak"]
TEMPLATES = [
    "The system feels {core} today.",
    "This pipeline is {core} for production.",
    "Users reported a {core} experience.",
    "The model shows {core} behavior in tests.",
]


def build_text_dataset(cfg: ExperimentConfig) -> tuple[list[str], np.ndarray]:
    texts: list[str] = []
    labels = np.zeros(cfg.num_samples, dtype=np.int64)
    for idx in range(cfg.num_samples):
        is_positive = idx < cfg.num_samples // 2
        core = np.random.choice(POSITIVE_TERMS if is_positive else NEGATIVE_TERMS)
        template = np.random.choice(TEMPLATES)
        texts.append(template.format(core=core))
        labels[idx] = int(is_positive)
    return texts, labels


texts_all, y_all = build_text_dataset(CFG)
split = int(0.8 * CFG.num_samples)
texts_train, texts_test = texts_all[:split], texts_all[split:]
y_train, y_test = y_all[:split], y_all[split:]
len(texts_train), len(texts_test), y_train.shape, y_test.shape

## 4) EDA

In [ ]:
train_lengths = np.array([len(text.split()) for text in texts_train], dtype=np.int64)
class_counts = np.bincount(y_all, minlength=CFG.num_classes)

print("Class counts:", class_counts.tolist())
print(
    "Train token-length mean/std:",
    float(train_lengths.mean()),
    float(train_lengths.std()),
)
print("Sample training sentence:", texts_train[0])

## 5) Tokenization and Preprocessing

In [ ]:
def normalize_text(text: str) -> str:

    return " ".join(text.lower().strip().split())





def tokenize_text(text: str) -> list[str]:

    return normalize_text(text).split()


In [ ]:
def build_vocabulary(train_texts: list[str], max_vocab_size: int = 2048) -> dict[str, int]:

    token_counts: dict[str, int] = {}

    for text in train_texts:

        for token in tokenize_text(text):

            token_counts[token] = token_counts.get(token, 0) + 1



    sorted_items = sorted(token_counts.items(), key=lambda item: (-item[1], item[0]))

    vocab: dict[str, int] = {"<PAD>": 0, "<UNK>": 1}

    for token, _ in sorted_items:

        if len(vocab) >= max_vocab_size:

            break

        vocab[token] = len(vocab)

    return vocab





def encode_and_pad(texts: list[str], vocab: dict[str, int], seq_len: int) -> np.ndarray:

    encoded = np.zeros((len(texts), seq_len), dtype=np.int64)

    unk_id = vocab["<UNK>"]

    for row_idx, text in enumerate(texts):

        token_ids = [vocab.get(token, unk_id) for token in tokenize_text(text)]

        trunc = token_ids[:seq_len]

        encoded[row_idx, : len(trunc)] = np.asarray(trunc, dtype=np.int64)

    return encoded





def token_ids_to_embeddings(token_ids: np.ndarray, vocab_size: int, d_model: int) -> np.ndarray:

    emb_table = np.random.normal(0.0, 0.2, (vocab_size, d_model)).astype(np.float32)

    return emb_table[token_ids]





def project_qkv(x: np.ndarray, d_model: int) -> tuple[np.ndarray, np.ndarray, np.ndarray]:

    w_q = np.random.normal(0.0, 0.2, (d_model, d_model)).astype(np.float32)

    w_k = np.random.normal(0.0, 0.2, (d_model, d_model)).astype(np.float32)

    w_v = np.random.normal(0.0, 0.2, (d_model, d_model)).astype(np.float32)

    q = x @ w_q

    k = x @ w_k

    v = x @ w_v

    return q, k, v





vocab = build_vocabulary(texts_train)

train_token_ids = encode_and_pad(texts_train, vocab, CFG.seq_len)

test_token_ids = encode_and_pad(texts_test, vocab, CFG.seq_len)



x_train = token_ids_to_embeddings(train_token_ids, len(vocab), CFG.d_model)

x_test = token_ids_to_embeddings(test_token_ids, len(vocab), CFG.d_model)

x_all = np.concatenate([x_train, x_test], axis=0)



q_train, k_train, v_train = project_qkv(x_train, CFG.d_model)

q_test, k_test, v_test = project_qkv(x_test, CFG.d_model)



print(

    {

        "vocab_size": len(vocab),

        "train_token_ids_shape": tuple(train_token_ids.shape),

        "embedding_shape": tuple(x_train.shape),

    }

)



# 6) Model Definition



def softmax(x: np.ndarray, axis: int = -1) -> np.ndarray:

    z = x - x.max(axis=axis, keepdims=True)

    exp_z = np.exp(z)

    return exp_z / exp_z.sum(axis=axis, keepdims=True)





def scaled_dot_product_attention(

    q: np.ndarray,

    k: np.ndarray,

    v: np.ndarray,

) -> tuple[np.ndarray, np.ndarray]:

    scores = (q @ np.swapaxes(k, -1, -2)) / math.sqrt(q.shape[-1])

    weights = softmax(scores, axis=-1)

    output = weights @ v

    return output, weights



# 7) Training



def pooled_features(attn_out: np.ndarray) -> np.ndarray:

    return attn_out.mean(axis=1)





attn_train, train_weights = scaled_dot_product_attention(q_train, k_train, v_train)

attn_test, test_weights = scaled_dot_product_attention(q_test, k_test, v_test)



x_feat_train = pooled_features(attn_train)

x_feat_test = pooled_features(attn_test)



w = np.zeros((CFG.d_model, 1), dtype=np.float32)

b = np.zeros((1,), dtype=np.float32)

y_train_f = y_train.astype(np.float32).reshape(-1, 1)



for _ in range(CFG.epochs):

    logits = x_feat_train @ w + b

    probs = 1.0 / (1.0 + np.exp(-logits))

    error = probs - y_train_f

    grad_w = (x_feat_train.T @ error) / x_feat_train.shape[0]

    grad_b = error.mean(axis=0)

    w -= CFG.learning_rate * grad_w

    b -= CFG.learning_rate * grad_b



print("Training finished.")



# 8) Evaluation and Metrics



def evaluate_binary_classifier(

    x_feat: np.ndarray,

    y_true: np.ndarray,

    w: np.ndarray,

    b: np.ndarray,

) -> dict[str, float]:

    logits = x_feat @ w + b

    probs = 1.0 / (1.0 + np.exp(-logits))

    preds = (probs >= 0.5).astype(np.int64).reshape(-1)

    accuracy = float((preds == y_true).mean())

    return {"accuracy": accuracy}





metrics = evaluate_binary_classifier(x_feat_test, y_test, w, b)

print(metrics)



# 9) Results Visualization



fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].imshow(test_weights[0], aspect="auto", cmap="viridis")

axes[0].set_title("Scaled Dot-Product Attention")

axes[1].bar(["accuracy"], [metrics["accuracy"]], color="#4C78A8")

axes[1].set_ylim(0.0, 1.0)

axes[1].set_title("Test Accuracy")

for ax in axes:

    ax.set_xlabel("Key position")

    ax.set_ylabel("Query position")

plt.tight_layout()

plt.show()



# 10) Summary

print("- Implemented scaled dot-product attention as the core similarity mechanism.")

print("- Trained a toy binary classifier on pooled attention features to complete the pipeline.")

print("- Mini Enterprise Use: semantic search and retrieval ranking services often start with this attention pattern.")
